<img src="../../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Lab: Cleaning Rock Song Data - Solution
---

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

### 1. Load `rock.csv` and do an initial examination of its data columns.

In [ ]:
rockfile = "../data/rock.csv"

In [ ]:
# Load the data.
df = pd.read_csv(rockfile)
df.head()

In [ ]:
# Look at the information regarding its columns.
df.info()

### 2.  Clean up the column names.

Let's clean up the column names. There are two ways we can accomplish this:

#### 2.A Change the column names when you import the data using `pd.read_csv()`.

Notice that, when passing `names=[..A LIST OF STRING..]` with a number of columns that matches the number of strings in the passed list, you replace the column names.

NOTE: When you create custom column names, the first row of the `.csv` already represents a header. It is important to tell `pandas` to skip that row. The `skiprows=1` keyword argument to `read_csv()` will tell `pandas` to skip the first row.

In [ ]:
# Change the column names when loading the '.csv':
column_names = ['song', 'artist', 'release', 'song_artist', 'first', 'year', 'playcount', 'fg']
df = pd.read_csv(rockfile, names=column_names, skiprows=1)
df.head()

#### 2.B Change column names using the `.rename()` function.

The `.rename()` function takes an argument, `columns=name_dict`, in which `name_dict` is a dictionary containing the original column names as keys and the new column names as values.

In [ ]:
# Change the column names using the `.rename()` function.

df = pd.read_csv(rockfile)

rename_map = {
    # Original column: [renamed column]
    'Song Clean':    'song', 
    'ARTIST CLEAN':  'artist', 
    'Release Year':  'release', 
    'COMBINED':      'song_artist', 
    'First?':        'first', 
    'Year?':         'year', 
    'PlayCount':     'playcount', 
    'F*G':           'fg'
}

df.rename(columns=rename_map, inplace=True)
df.head(4)

#### 2.C Reassigning the `.columns` attribute of a DataFrame.

You can also just reassign the `.columns` attribute to a list of strings containing the new column names. 

The only caveat with reassigning `.columns` is that you have to reassign all of the column names at once. You can't partially replace a value by working on `.columns` directly. You have to reassign `.columns` with a list of equal length. 

In [ ]:
# Replace the column names by reassigning the `.columns` attribute.
df = pd.read_csv(rockfile)
column_names = ['song', 'artist', 'release', 'song_artist', 'first', 'year', 'playcount', 'fg']
df.columns = column_names
df.head()

### 3. Subsetting data where null values exist.

We have mixed `str` and `NaN` values in the `release` column. `NaN` stands for "not a number" and is the way `pandas` handles "nulls" or nonexistent data. We can use the `.isnull()` method of a Series to find null values.

Print the header of the data subset to where the `release` column is null values.

In [ ]:
# This will show us records where `df['release']` is null.
null_release_mask = df['release'].isnull()
df[null_release_mask].head()

### 4. Update slices of your DataFrame based on mask selection/slices.

In many scenarios, we want to upate values in our DataFrame according to criteria. Let's say we wanted to set all of the null values in `release` to 0.

With newer versions of `pandas`, in order to manipulate data in the original DataFrame, we have to use `.loc` while performing reassignment using a mask and an index.

For example, the following won't always work:
```python
df[row_mask]['column_name'] = new_value
```

The best way to accomplish the same task is:
```python
df.loc[row_mask, 'column_name'] = new_value
```

For multiple column assignment, you would use:
```python
df.loc[row_mask, ['col_1', 'col_2', 'col_3']] = new_value
```

#### 4.A Let's try it out. Make all of the null values in `release` 0.

In [ ]:
# We're going to reload our data to a fresh state.
column_names = ['song', 'artist', 'release', 'song_artist', 'first', 'year', 'playcount', 'fg']
df = pd.read_csv(rockfile, names=column_names, skiprows=1)

# create/identify the target subset and use it as an indexing tool to set values.
null_release_mask = df['release'].isnull()
df.loc[null_release_mask, 'release'] = 0

# We'll then print out our DataFrame's first 15 rows:
df.head(15)

#### 4.B Verify that `release` contains no null values.

In [ ]:
df.isnull().sum()

### 5. Ensure that the data types of the columns make sense. 

Verifying column data types is a critical part of data munging. If columns have the wrong data type, then there is usually corrupted or incorrect data in some of the observations.

#### 5.A Look at the data types for the columns. Are any incorrect given what the data represents?

In [ ]:
# We're going to reload our data to a fresh state.
column_names = ['song', 'artist', 'release', 'song_artist', 'first', 'year', 'playcount', 'fg']
df = pd.read_csv(rockfile, names=column_names, skiprows=1)

df.info()

In [ ]:
df.head(3)

_Only the `release` column appears to be wrong. It is represented as a string but should be an integer for year._

### 6. Investigate and clean up the `release` column.

The `release` column is a string data type when it should be an integer.

#### 6.A Figure out what value(s) are causing the `release` column to be encoded as a string instead of an integer.

In [ ]:
# Looking at the unique values in the column can be a good way to find offending values:
df.release.unique()

In [ ]:
# A row has SONGFACTS.COM as a value — this is clearly not a year.

#### 6.B Look at the rows in which there is incorrect data in the `release` column.

In [ ]:
# Slice and assign.
release_mask = df['release'] == "SONGFACTS.COM"
df[release_mask]

#### 6.C. Clean up the data. Normally we may replace the offending data with null np.nan values, however we previously converted all of the nan values in the release column to zeros so we might as well continue with the same practice. Replacing with 0 (or nan) will allow us to convert the column to numeric.

In [ ]:
df.loc[release_mask, 'release'] = np.nan

df = df.astype({'release': 'float'}) # you can use astype() to convert a column's data type

In [ ]:
df.info()

**Note:** Year can also be considered a descriptive value and therefore it also makes sense for a Year column to be an object.  
However, just like in this situation, using conversion to numerics is a great way of identifying improper values in a year column.

### 7. Get summary statistics for the `release` column using the `.describe()` function.

Now that the `release` column is finally a numeric data type, we can apply the `.describe()` function.  

#### 7.A Print out the summary stats for the `release` column. What is the earliest and latest release date?

In [ ]:
df['release'].describe()

In [ ]:
# The earliest release date is is 1071 and latest is 2014.

#### 7.B Based on the summary statistics, is there anything else wrong with the `release` column? 

In [ ]:
# A year of 1071 for a song release seems wrong. 
# We might want to impose a cut off for what the earliest song can be.
df[df.release == 1071]

_Looking at the DataFrame that contains the year 1071, we can see that the year was probably corrupted and should be replaced with something else if possible._

### *Stretch: we have not yet covered `.apply()`.  See if you can work out this part.*

### 8. Make changes and investigate using custom functions with `.apply()`.

Let's say we want to traverse every single row in our data set and apply a function to that row.

#### 8.A Write a function that will take a row of a DataFrame and print out the song, artist, and whether or not the release date is < 1970.


In [ ]:
def release_inspector(row):
    print('--------------------------------')
    print(row['song'], row['artist'], row['release'], '< 1970?:', row['release'] < 1970)

#### 8.B Using the `.apply()` function, apply the function you wrote to the first four rows of the DataFrame.

You will need to tell the `apply` function to operate row by row. Setting the keyword argument as `axis=1` indicates that the function should be applied to each row individually.

In [ ]:
df.head(4).apply(release_inspector, axis=1)

You'll notice that there will be a final output Series of `None` values. The `.apply()` function, if a return value is not specified, will return a Series of `None` values (similar to how the default return for Python functions is `None` when a return statement is not specified).

### 9. Write a function that converts cells in a DataFrame to float and otherwise replaces them with `np.nan`.

If applied to our data, it would keep only the numeric information and otherwise input null values.

Recall that the try-except syntax in Python is a great way to try something and take another action if the initial step fails:

```python
try:
    Perform some action.
except:
   Perform some other action if the first failed with an error.
```

#### 9.A Write the function that takes a column and converts all of its values to float if possible and `np.nan` otherwise. The return value should be the converted Series.

In [ ]:
def converter_helper(value):
    try:
        return float(value)
    except:
        return np.nan

def convert_to_float(column):
    column = column.map(converter_helper)
    return column

#### 9.B Try your function out on the rock song data and ensure the output is what you expected.


In [ ]:
df.apply(convert_to_float).head(10)

In [ ]:
# create a new dataframe via applying our function

df2 = df.apply(convert_to_float)

#### 9.C Describe the new float-only DataFrame.

In [ ]:
df2.describe()